# Simulation 1 Monitoring Cued Retrieval Figure

> Generate a current-regime follow-up that adds target monitoring to the VIT-style cued retrieval setup.

This notebook crosses reminder and task encoding conditions under supplied film cues, then tunes start drift and off-target updating to evaluate whether monitoring reduces a maximal interference contrast.

In [ ]:
# --- sweep configuration ---
PROJECT_ROOT = ""
FIT_PATH = "results/fits/Dupertuys2026_eCMR_source_only_phi_no_shared_support_best_of_1.json"
FIGURE_DIR = "figures/exploratory"
DATA_DIR = "figure_data/exploratory"
FIGURE_STR = "simulation2_control_cued"
RNG_SEED = 0

TASK_MCF_VALUES = [1.0, 2.0]
TASK_MCF_LABELS = {
    1.0: "Weak task encoding",
    2.0: "Strong task encoding",
}

PRIMARY_SOURCE_ORIENTATION = 0.0

BASELINE_START_DRIFT = 0.0
START_DRIFT_CANDIDATES = [1.0, 1.5, 2.0, 2.5]
KAPPA_VALUES = [1.0, 0.75, 0.5, 0.25, 0.0]
SELECTED_START_DRIFT_OVERRIDE = None
SELECTED_KAPPA_OVERRIDE = None

FILM_CUE_REINSTATEMENT = 0.90
CUE_INTERVAL = 4
FIRST_CUE_AFTER = 0
SAVE_BEHAVIORAL_FIGURE = False
SAVE_MECHANISM_FIGURE = False

N_FILM = 16
N_BREAK = 16
N_INTERFERENCE = 16
N_FILLER = 16
EXPERIMENT_COUNT = 5000
MAX_RECALL = 48

FILM_EMOTIONAL = True
INTERFERENCE_EMOTIONAL = False
CACHE_AFTER = "filler"

EXTRA_FIXED_SCALES = {
    "source_learning_baseline": 0.05,
    "primacy_scale": 0.1,
    "primacy_decay": 1.5,
}

BASE_FIXED_SCALES = {
    "break_drift_scale": 1.0,
    "break_mcf_scale": 1.0,
    "filler_drift_scale": 1.0,
    "filler_mcf_scale": 1.0,
}

REMINDER_CONDITIONS = {
    "No reminder": {
        "reminder_start_drift_scale": 0.0,
        "reminder_drift_scale": 0.0,
    },
    "With reminder": {
        "reminder_start_drift_scale": 1.0,
        "reminder_drift_scale": 1.0,
    },
}

RETRIEVAL_FIXED_SCALES = {
    "tau_scale": 1.0,
    "target_recall_drift_scale": 1.0,
    "recall_drift_scale": 1.0,
}

In [ ]:
import csv
import os
import warnings
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown
from jax import random
from jaxcmr.analyses.spc import fixed_pres_spc
from jaxcmr.helpers import find_project_root

from selective_interference_v2 import (
    PHASE_COLORS,
    Paradigm,
    add_phase_bands,
    apply_supplied_item_context_cue,
    batch_trial,
    configure_rates,
    light_to_dark_colors,
    load_fit_params,
    make_factory,
    make_is_emotional,
    make_is_target,
    prepare_sweep,
    save_figure,
    simulate_periodic_cued_free_recall,
    split_scales_for_cache,
    standard_remap,
    sweep_rngs,
)

warnings.filterwarnings("ignore")

In [ ]:
#| output: asis
if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT)
else:
    project_root = Path(find_project_root())

fit_path = project_root / FIT_PATH
figure_dir = project_root / FIGURE_DIR if FIGURE_DIR else None
data_dir = project_root / DATA_DIR if DATA_DIR else None
task_mcf_values = np.asarray(TASK_MCF_VALUES, dtype=float)
start_drift_values = np.asarray([BASELINE_START_DRIFT] + START_DRIFT_CANDIDATES, dtype=float)
kappa_values = np.asarray(KAPPA_VALUES, dtype=float)
extra_fixed_scales = {**EXTRA_FIXED_SCALES}
base_fixed_scales = {**BASE_FIXED_SCALES, **extra_fixed_scales}

params, n_subjects = load_fit_params(fit_path)
paradigm = Paradigm(
    n_film=N_FILM,
    n_break=N_BREAK,
    n_interference=N_INTERFERENCE,
    n_filler=N_FILLER,
    experiment_count=EXPERIMENT_COUNT,
    max_recall=MAX_RECALL,
)
factory = make_factory(
    is_emotional=make_is_emotional(
        paradigm,
        film_emotional=FILM_EMOTIONAL,
        interference_emotional=INTERFERENCE_EMOTIONAL,
    ),
    is_target=make_is_target(paradigm),
)
rng = random.PRNGKey(RNG_SEED)

Markdown(
    f"Loaded {n_subjects} fit parameter set(s). Running {len(start_drift_values)} start-drift values, "
    f"{len(kappa_values)} monitoring values, {len(REMINDER_CONDITIONS)} reminder conditions, "
    f"and {len(task_mcf_values)} task encoding values."
)

In [ ]:
def position_phase_labels(paradigm):
    return (
        ["film"] * paradigm.n_film
        + ["break"] * paradigm.n_break
        + ["task"] * paradigm.n_interference
        + ["filler"] * paradigm.n_filler
    )


def write_rows(path, fieldnames, rows):
    os.makedirs(path.parent, exist_ok=True)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def phase_recall_totals(spc):
    totals = {}
    for phase in sorted(set(position_phase)):
        positions = [idx for idx, value in enumerate(position_phase) if value == phase]
        totals[phase] = float(np.sum(spc[positions]))
    return totals


def off_target_film_return_probability(recalls, n_film):
    numerator = 0
    denominator = 0
    for row in recalls:
        sampled = [int(value) for value in row if value > 0]
        seen_film = set()
        for idx in range(len(sampled) - 1):
            current = sampled[idx]
            next_sample = sampled[idx + 1]
            if 1 <= current <= n_film:
                seen_film.add(current)
                continue
            if len(seen_film) >= n_film:
                continue
            denominator += 1
            if 1 <= next_sample <= n_film:
                numerator += 1
                seen_film.add(next_sample)
    probability = np.nan if denominator == 0 else numerator / denominator
    return probability, denominator, numerator


position_phase = position_phase_labels(paradigm)
n_presented = len(position_phase)

In [ ]:
def cued_recall_trial(
    model,
    rng,
    film_items,
    cue_scale,
    max_recall,
    cue_interval,
    first_cue_after,
):
    recall_rng, initial_cue_rng = random.split(rng)
    model = model.start_retrieving()
    if first_cue_after == 0:
        cue_item = random.choice(initial_cue_rng, film_items)
        model = apply_supplied_item_context_cue(model, cue_item, cue_scale)
        scheduled_first_cue_after = cue_interval
    else:
        scheduled_first_cue_after = first_cue_after
    _, recalls, cue_attempts, cue_items = simulate_periodic_cued_free_recall(
        model,
        max_recall,
        recall_rng,
        film_items,
        cue_scale,
        cue_interval=cue_interval,
        first_cue_after=scheduled_first_cue_after,
    )
    return recalls, cue_attempts, cue_items


batched_cued_recall = batch_trial(cued_recall_trial, n_args=7, n_static=3)

prepared_cache = {}
for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
    for task_scale in task_mcf_values:
        fixed_scales = {
            **base_fixed_scales,
            **reminder_scales,
            "interference_mcf_scale": float(task_scale),
        }
        pre_cache_scales, post_cache_scales = split_scales_for_cache(
            fixed_scales,
            CACHE_AFTER,
        )
        prepared = prepare_sweep(
            params,
            paradigm,
            factory,
            cache_after=CACHE_AFTER,
            **pre_cache_scales,
        )
        prepared_cache[(reminder_condition, float(task_scale))] = prepared

In [ ]:
phase_rows = []
spc_rows = []
recovery_rows = []
spc_curves = {}
cued_rng = random.PRNGKey(RNG_SEED + 2)

for start_drift_scale in start_drift_values:
    for kappa in kappa_values:
        for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
            for task_scale in task_mcf_values:
                fixed_scales = {
                    **base_fixed_scales,
                    **reminder_scales,
                    "interference_mcf_scale": float(task_scale),
                }
                _, post_cache_scales = split_scales_for_cache(
                    fixed_scales,
                    CACHE_AFTER,
                )
                prepared = prepared_cache[(reminder_condition, float(task_scale))]
                ready_models = configure_rates(
                    prepared.models,
                    **post_cache_scales,
                    **RETRIEVAL_FIXED_SCALES,
                    start_drift_scale=float(start_drift_scale),
                    rejected_recall_drift_scale=float(kappa),
                    film_source_start_drift_rate=float(PRIMARY_SOURCE_ORIENTATION),
                )
                rngs, cued_rng = sweep_rngs(
                    cued_rng,
                    prepared.n_subjects,
                    prepared.experiment_count,
                )
                recalls, cue_attempts, cue_items = batched_cued_recall(
                    ready_models,
                    rngs,
                    paradigm.film_items,
                    FILM_CUE_REINSTATEMENT,
                    paradigm.max_recall,
                    CUE_INTERVAL,
                    FIRST_CUE_AFTER,
                )
                raw_recalls = np.asarray(recalls).reshape(-1, recalls.shape[-1])
                remapped_recalls = np.asarray(
                    standard_remap(
                        raw_recalls,
                        paradigm,
                        show_break=True,
                    )
                )
                spc = np.asarray(fixed_pres_spc(remapped_recalls, n_presented), dtype=float)
                totals = phase_recall_totals(spc)
                recovery, denominator, numerator = off_target_film_return_probability(
                    remapped_recalls,
                    paradigm.n_film,
                )
                key = (
                    float(start_drift_scale),
                    float(kappa),
                    reminder_condition,
                    float(task_scale),
                )
                spc_curves[key] = spc
                for phase, total in totals.items():
                    phase_rows.append({
                        "start_drift_scale": float(start_drift_scale),
                        "rejected_recall_drift_scale": float(kappa),
                        "reminder_condition": reminder_condition,
                        "task_condition": TASK_MCF_LABELS[float(task_scale)],
                        "task_mcf_scale": float(task_scale),
                        "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                        "first_cue_after": FIRST_CUE_AFTER,
                        "film_source_start_drift_rate": float(PRIMARY_SOURCE_ORIENTATION),
                        "phase": phase,
                        "recall_probability_mass": total,
                    })
                for position, probability in enumerate(spc, start=1):
                    spc_rows.append({
                        "start_drift_scale": float(start_drift_scale),
                        "rejected_recall_drift_scale": float(kappa),
                        "reminder_condition": reminder_condition,
                        "task_condition": TASK_MCF_LABELS[float(task_scale)],
                        "task_mcf_scale": float(task_scale),
                        "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                        "first_cue_after": FIRST_CUE_AFTER,
                        "film_source_start_drift_rate": float(PRIMARY_SOURCE_ORIENTATION),
                        "position": position,
                        "phase": position_phase[position - 1],
                        "recall_probability": float(probability),
                    })
                recovery_rows.append({
                    "start_drift_scale": float(start_drift_scale),
                    "rejected_recall_drift_scale": float(kappa),
                    "reminder_condition": reminder_condition,
                    "task_condition": TASK_MCF_LABELS[float(task_scale)],
                    "task_mcf_scale": float(task_scale),
                    "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                    "first_cue_after": FIRST_CUE_AFTER,
                    "film_source_start_drift_rate": float(PRIMARY_SOURCE_ORIENTATION),
                    "film_return_probability": recovery,
                    "transition_count": denominator,
                    "film_return_count": numerator,
                })

In [ ]:
def phase_total(start_drift_scale, kappa, reminder_condition, task_scale, phase):
    for row in phase_rows:
        if (
            np.isclose(row["start_drift_scale"], start_drift_scale)
            and np.isclose(row["rejected_recall_drift_scale"], kappa)
            and row["reminder_condition"] == reminder_condition
            and np.isclose(row["task_mcf_scale"], task_scale)
            and row["phase"] == phase
        ):
            return row["recall_probability_mass"]
    return np.nan


def maximal_interference_cost(start_drift_scale, kappa):
    benign = phase_total(
        start_drift_scale,
        kappa,
        "No reminder",
        float(TASK_MCF_VALUES[0]),
        "film",
    )
    interference = phase_total(
        start_drift_scale,
        kappa,
        "With reminder",
        float(TASK_MCF_VALUES[-1]),
        "film",
    )
    cost = benign - interference
    percent = cost / benign if benign > 0 else np.nan
    return benign, interference, cost, percent


baseline_benign, baseline_hardest, baseline_cost, baseline_percent = maximal_interference_cost(
    BASELINE_START_DRIFT,
    1.0,
)
selection_rows = []
viable_rows = []
for start_drift_scale in START_DRIFT_CANDIDATES:
    for kappa in KAPPA_VALUES:
        benign, hardest, cost, percent = maximal_interference_cost(start_drift_scale, kappa)
        cost_reduction = baseline_cost - cost
        hardest_gain = hardest - baseline_hardest
        is_monitoring = kappa < 1.0
        is_viable = bool(is_monitoring and cost < baseline_cost and hardest > baseline_hardest)
        row = {
            "start_drift_scale": float(start_drift_scale),
            "rejected_recall_drift_scale": float(kappa),
            "benign_film_recall": benign,
            "hardest_film_recall": hardest,
            "maximal_interference_cost": cost,
            "maximal_interference_percent": percent,
            "cost_reduction_vs_unguided": cost_reduction,
            "hardest_gain_vs_unguided": hardest_gain,
            "is_viable": is_viable,
        }
        selection_rows.append(row)
        if is_viable:
            viable_rows.append(row)

if SELECTED_START_DRIFT_OVERRIDE is not None and SELECTED_KAPPA_OVERRIDE is not None:
    selected_start_drift = float(SELECTED_START_DRIFT_OVERRIDE)
    selected_kappa = float(SELECTED_KAPPA_OVERRIDE)
    selection_reason = "manual override"
elif viable_rows:
    viable_rows = sorted(
        viable_rows,
        key=lambda row: (
            row["start_drift_scale"],
            -row["rejected_recall_drift_scale"],
        ),
    )
    selected_start_drift = viable_rows[0]["start_drift_scale"]
    selected_kappa = viable_rows[0]["rejected_recall_drift_scale"]
    selection_reason = "smallest start drift and largest kappa reducing absolute cost"
else:
    monitoring_rows = [row for row in selection_rows if row["rejected_recall_drift_scale"] < 1.0]
    monitoring_rows = sorted(
        monitoring_rows,
        key=lambda row: (
            row["maximal_interference_cost"],
            row["start_drift_scale"],
            -row["rejected_recall_drift_scale"],
        ),
    )
    selected_start_drift = monitoring_rows[0]["start_drift_scale"]
    selected_kappa = monitoring_rows[0]["rejected_recall_drift_scale"]
    selection_reason = "no viable absolute-cost reduction; chose smallest observed monitoring cost"

RETRIEVAL_REGIMES = {
    "Unguided recall": {
        "start_drift_scale": float(BASELINE_START_DRIFT),
        "rejected_recall_drift_scale": 1.0,
    },
    "Deliberate recall: start context": {
        "start_drift_scale": float(selected_start_drift),
        "rejected_recall_drift_scale": 1.0,
    },
    "Deliberate recall: start context + monitoring": {
        "start_drift_scale": float(selected_start_drift),
        "rejected_recall_drift_scale": float(selected_kappa),
    },
}

cost_rows = []
for regime, scales in RETRIEVAL_REGIMES.items():
    benign, hardest, cost, percent = maximal_interference_cost(
        scales["start_drift_scale"],
        scales["rejected_recall_drift_scale"],
    )
    cost_rows.append({
        "retrieval_regime": regime,
        "start_drift_scale": scales["start_drift_scale"],
        "rejected_recall_drift_scale": scales["rejected_recall_drift_scale"],
        "benign_film_recall": benign,
        "hardest_film_recall": hardest,
        "maximal_interference_cost": cost,
        "maximal_interference_percent": percent,
        "selection_reason": selection_reason if regime == "Deliberate recall: start context + monitoring" else "",
    })

Markdown(
    f"Selected start drift {selected_start_drift:.2f} and kappa {selected_kappa:.2f}: {selection_reason}."
)

In [ ]:
if data_dir is not None and FIGURE_STR:
    write_rows(
        data_dir / f"{FIGURE_STR}_phase_totals.csv",
        [
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "film_source_start_drift_rate",
            "phase",
            "recall_probability_mass",
        ],
        phase_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_spc.csv",
        [
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "film_source_start_drift_rate",
            "position",
            "phase",
            "recall_probability",
        ],
        spc_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_offtarget_recovery.csv",
        [
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "film_source_start_drift_rate",
            "film_return_probability",
            "transition_count",
            "film_return_count",
        ],
        recovery_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_diagnostic_grid.csv",
        [
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "benign_film_recall",
            "hardest_film_recall",
            "maximal_interference_cost",
            "maximal_interference_percent",
            "cost_reduction_vs_unguided",
            "hardest_gain_vs_unguided",
            "is_viable",
        ],
        selection_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_maximal_interference_cost.csv",
        [
            "retrieval_regime",
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "benign_film_recall",
            "hardest_film_recall",
            "maximal_interference_cost",
            "maximal_interference_percent",
            "selection_reason",
        ],
        cost_rows,
    )

In [ ]:
PANEL_LETTER_FONTSIZE = 16
PANEL_TITLE_FONTSIZE = 13
STRUCTURAL_FONTSIZE = 11
SUPPORT_FONTSIZE = 10

BAR_REMINDER_LABELS = {
    "No reminder": "No reminder",
    "With reminder": "Film reminder",
}
FILM_BAR_COLOR = PHASE_COLORS["film"]
FILM_BAR_PALE = "#E8F1FF"
FILM_BAR_EDGE = PHASE_COLORS["film"]

COST_TICK_LABELS = {
    "Unguided recall": "Unguided\nrecall",
    "Deliberate recall: start context": "Start\ncontext",
    "Deliberate recall: start context + monitoring": "Start context\n+ monitoring",
}


def add_panel_label(axis, label, x=-0.16, y=1.12):
    axis.text(
        x,
        y,
        label,
        transform=axis.transAxes,
        fontsize=PANEL_LETTER_FONTSIZE,
        fontweight="bold",
        va="top",
        ha="left",
    )


def regime_total(regime, reminder_condition, task_scale, phase):
    scales = RETRIEVAL_REGIMES[regime]
    return phase_total(
        scales["start_drift_scale"],
        scales["rejected_recall_drift_scale"],
        reminder_condition,
        task_scale,
        phase,
    )


def plot_2x2_film_recall(axis, regime, y_upper=None, show_legend=False):
    x = np.arange(len(task_mcf_values))
    width = 0.34
    offsets = [-width / 2, width / 2]
    for offset, reminder_condition in zip(offsets, REMINDER_CONDITIONS):
        values = [
            regime_total(
                regime,
                reminder_condition,
                float(task_scale),
                "film",
            )
            for task_scale in task_mcf_values
        ]
        if reminder_condition == "No reminder":
            facecolor = FILM_BAR_PALE
            edgecolor = FILM_BAR_EDGE
            linewidth = 1.1
        else:
            facecolor = FILM_BAR_COLOR
            edgecolor = FILM_BAR_EDGE
            linewidth = 1.1
        axis.bar(
            x + offset,
            values,
            width=width,
            color=facecolor,
            edgecolor=edgecolor,
            linewidth=linewidth,
            label=BAR_REMINDER_LABELS[reminder_condition],
            zorder=2,
        )
    title_labels = {
        "Unguided recall": "Unguided recall",
        "Deliberate recall: start context": "Deliberate recall:\nstart context",
        "Deliberate recall: start context + monitoring": "Deliberate recall:\nstart context + monitoring",
    }
    axis.set_title(title_labels[regime], fontsize=PANEL_TITLE_FONTSIZE, pad=8)
    axis.set_xticks(x)
    task_tick_labels = {
        "Weak task encoding": "Weak task\nencoding",
        "Strong task encoding": "Strong task\nencoding",
    }
    axis.set_xticklabels([task_tick_labels[TASK_MCF_LABELS[float(value)]] for value in task_mcf_values])
    axis.set_ylim(0, y_upper)
    axis.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="#E7EBF0", linewidth=0.45, alpha=0.45, zorder=0)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if show_legend:
        axis.legend(
            frameon=False,
            fontsize=SUPPORT_FONTSIZE,
            loc="upper right",
            borderaxespad=0.3,
        )


film_values = [
    regime_total(regime, reminder_condition, float(task_scale), "film")
    for regime in RETRIEVAL_REGIMES
    for reminder_condition in REMINDER_CONDITIONS
    for task_scale in task_mcf_values
]
y_upper = max(1.0, np.ceil(max(film_values) * 1.14 * 4) / 4)

fig = plt.figure(figsize=(11.0, 6.45))
gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.68], hspace=0.58, wspace=0.42)
raw_axes = [fig.add_subplot(gs[0, idx]) for idx in range(3)]
bottom_gs = gs[1, :].subgridspec(1, 3, width_ratios=[0.16, 0.68, 0.16])
axis_cost = fig.add_subplot(bottom_gs[0, 1])

for label, axis, regime in zip(["A", "B", "C"], raw_axes, RETRIEVAL_REGIMES):
    plot_2x2_film_recall(
        axis,
        regime,
        y_upper=y_upper,
        show_legend=label == "C",
    )
    add_panel_label(axis, label)
raw_axes[0].set_ylabel("Mean film items recalled", fontsize=STRUCTURAL_FONTSIZE)

cost_x = np.arange(len(cost_rows))
cost_values = [row["maximal_interference_cost"] for row in cost_rows]
axis_cost.bar(
    cost_x,
    cost_values,
    width=0.48,
    color=FILM_BAR_COLOR,
    edgecolor=FILM_BAR_EDGE,
    linewidth=1.1,
    zorder=2,
)
axis_cost.set_title("Film output reduction", fontsize=PANEL_TITLE_FONTSIZE, pad=8)
axis_cost.set_xticks(cost_x)
axis_cost.set_xticklabels([COST_TICK_LABELS[row["retrieval_regime"]] for row in cost_rows])
axis_cost.set_ylabel("Film items reduced", fontsize=STRUCTURAL_FONTSIZE)
axis_cost.set_ylim(0, max(cost_values) * 1.18)
axis_cost.tick_params(labelsize=SUPPORT_FONTSIZE)
axis_cost.grid(axis="y", color="#E7EBF0", linewidth=0.45, alpha=0.45, zorder=0)
axis_cost.spines["top"].set_visible(False)
axis_cost.spines["right"].set_visible(False)
add_panel_label(axis_cost, "D", x=-0.13, y=1.12)

if SAVE_BEHAVIORAL_FIGURE:
    save_figure(
        str(figure_dir) if figure_dir is not None else "",
        FIGURE_STR,
        "behavioral",
    )


In [ ]:
FOCUS_REMINDER_CONDITION = "With reminder"
FOCUS_TASK_SCALE = float(TASK_MCF_VALUES[-1])
focus_recovery_rows = [
    row
    for row in recovery_rows
    if (
        np.isclose(row["start_drift_scale"], selected_start_drift)
        and row["reminder_condition"] == FOCUS_REMINDER_CONDITION
        and np.isclose(row["task_mcf_scale"], FOCUS_TASK_SCALE)
    )
]
focus_recovery_rows = sorted(
    focus_recovery_rows,
    key=lambda row: row["rejected_recall_drift_scale"],
)
focus_kappas = [row["rejected_recall_drift_scale"] for row in focus_recovery_rows]
focus_recovery = [row["film_return_probability"] for row in focus_recovery_rows]
focus_spc_curves = [
    spc_curves[(selected_start_drift, kappa, FOCUS_REMINDER_CONDITION, FOCUS_TASK_SCALE)]
    for kappa in focus_kappas
]


def add_phase_boundaries(axis):
    boundaries = np.cumsum([
        paradigm.n_film,
        paradigm.n_break,
        paradigm.n_interference,
    ]) + 0.5
    for boundary in boundaries:
        axis.axvline(boundary, color="#7C8794", linewidth=0.7, alpha=0.45, zorder=1)


def add_reminder_marker(axis):
    reminder_x = paradigm.n_film + paradigm.n_break + 0.5
    axis.axvline(
        reminder_x,
        color=PHASE_COLORS["reminder"],
        linewidth=1.6,
        alpha=0.85,
        zorder=2,
    )
    axis.text(
        reminder_x - 1.05,
        0.62,
        "Film reminder",
        ha="center",
        va="center",
        fontsize=STRUCTURAL_FONTSIZE,
        fontstyle="italic",
        color=PHASE_COLORS["reminder"],
        rotation=90,
        rotation_mode="anchor",
        transform=axis.get_xaxis_transform(),
        clip_on=False,
        zorder=3,
    )


fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2), gridspec_kw={"width_ratios": [0.92, 1.35]})
axis_a, axis_b = axes

axis_a.plot(
    focus_kappas,
    focus_recovery,
    color="#222222",
    marker="o",
    linewidth=1.8,
)
axis_a.set_title("Return after off-target sample", fontsize=PANEL_TITLE_FONTSIZE, pad=8)
axis_a.set_xlabel("Off-target update (kappa)", fontsize=STRUCTURAL_FONTSIZE)
axis_a.set_ylabel("Probability next sample is film", fontsize=STRUCTURAL_FONTSIZE)
axis_a.set_xlim(-0.04, 1.04)
axis_a.set_ylim(0, max(focus_recovery) * 1.18 if focus_recovery else 1.0)
axis_a.tick_params(labelsize=SUPPORT_FONTSIZE)
axis_a.grid(axis="y", color="#E7EBF0", linewidth=0.45, alpha=0.45, zorder=0)
axis_a.spines["top"].set_visible(False)
axis_a.spines["right"].set_visible(False)
add_panel_label(axis_a, "A")

add_phase_bands(axis_b, position_phase, fontsize=STRUCTURAL_FONTSIZE)
add_phase_boundaries(axis_b)
add_reminder_marker(axis_b)
positions = np.arange(1, n_presented + 1)
colors = light_to_dark_colors(len(focus_kappas))
for color, kappa, curve in zip(colors, focus_kappas, focus_spc_curves):
    axis_b.plot(
        positions,
        curve,
        color=color,
        linewidth=1.5,
        label=f"{kappa:.2f}",
    )
axis_b.set_title("Overall recall", fontsize=PANEL_TITLE_FONTSIZE, pad=36)
axis_b.set_xlabel("Encoded position", fontsize=STRUCTURAL_FONTSIZE)
axis_b.set_ylabel("Recall probability", fontsize=STRUCTURAL_FONTSIZE)
axis_b.set_xlim(1, n_presented)
axis_b.set_ylim(-0.025, max(float(np.max(curve)) for curve in focus_spc_curves) * 1.16)
axis_b.tick_params(labelsize=SUPPORT_FONTSIZE)
axis_b.spines["top"].set_visible(False)
axis_b.spines["right"].set_visible(False)
axis_b.legend(
    title="Off-target\nupdate",
    frameon=True,
    fontsize=SUPPORT_FONTSIZE,
    title_fontsize=SUPPORT_FONTSIZE,
    loc="upper right",
)
add_panel_label(axis_b, "B")

if SAVE_MECHANISM_FIGURE:
    save_figure(
        str(figure_dir) if figure_dir is not None else "",
        FIGURE_STR,
        "mechanism_offtarget_monitoring",
    )

In [ ]:
print(f"Selected start drift: {selected_start_drift:.2f}")
print(f"Selected kappa: {selected_kappa:.2f}")
print(f"Selection reason: {selection_reason}")
print("Maximal contrast costs")
for row in cost_rows:
    print(
        f"  {row['retrieval_regime']}: "
        f"{row['benign_film_recall']:.3f} -> {row['hardest_film_recall']:.3f}, "
        f"drop={row['maximal_interference_cost']:.3f}, "
        f"pct={row['maximal_interference_percent'] * 100:.1f}%"
    )
print("Best diagnostic rows by absolute cost")
for row in sorted(selection_rows, key=lambda value: value["maximal_interference_cost"])[:8]:
    print(
        f"  start={row['start_drift_scale']:.2f}, kappa={row['rejected_recall_drift_scale']:.2f}: "
        f"drop={row['maximal_interference_cost']:.3f}, "
        f"pct={row['maximal_interference_percent'] * 100:.1f}%, "
        f"hardest={row['hardest_film_recall']:.3f}, viable={row['is_viable']}"
    )